# Embedding Model Benchmark**Runs on Colab or local Jupyter.** No GPU is required for this notebook (see below), so a local machine is perfectly fine here even without a GPU; Colab works too if that's more convenient for continuity with the generation notebook.**Purpose**: choose which embedding model to standardize on for the paradigm benchmark (pgvector vs MongoDB vs Qdrant), by comparing candidate embedding models on retrieval quality against your own W1/W2/W3 query sets.**Why this comparison has to happen before the paradigm benchmark**: the paradigm benchmark holds the embedding model fixed and varies the database. If the embedding model were chosen without evidence, a weak choice there would cap the ceiling of every downstream result regardless of which database performs best. This notebook removes that variable first.**Models compared**:| Model | Type | Dim ||---|---|---|| `text-embedding-3-small` | OpenAI API | 1536 || `text-embedding-3-large` | OpenAI API | 3072 || `BAAI/bge-small-en-v1.5` | open-source | 384 || `Alibaba-NLP/gte-base-en-v1.5` | open-source | 768 |**Why no GPU is needed here**: retrieval scoring is cosine similarity over already-computed embedding vectors plus rank-based metrics. Both are cheap array operations, so this runs fine on a local machine. GPU was only needed in the generation notebook, where raw text had to be encoded by the open-source models.**Data source**: everything is pulled from the private HF dataset `Sakhiur/empirical-rag-paradigm-benchmark` (queries CSVs + all four models' embedding jsonl files), so this notebook is reproducible from any machine without needing local files copied around.**Evaluation slices**: W1 x W2 x W3, crossed with abstracts x pdf_chunks, giving 6 separate benchmark runs. Keeping these separate rather than pooling matters because the three workload types test fundamentally different retrieval behavior (short factual lookup vs multi-hop reasoning vs semantic similarity), so pooling them would average away exactly the differences the thesis needs to report.

## SetupRuns on Colab or local Jupyter unchanged, since nothing in this notebook depends on Colab-only APIs: it's cosine similarity, ranking, and plotting, none of which needs GPU or Colab-specific secret handling. `%pip` (rather than `!pip`) is used so the install always targets the kernel actually running the notebook, which matters more on local Jupyter where multiple Python environments can be on the PATH at once.

In [ ]:
%pip install -q ranx rich huggingface_hub pandas numpy scikit-learn matplotlib sentence-transformers openai python-dotenv

In [ ]:
import osimport jsonimport stringimport mathfrom pathlib import Pathfrom typing import Anyimport numpy as npimport pandas as pdimport matplotlib.pyplot as pltfrom ranx import Qrels, Run, comparefrom ranx.data_structures.report import Reportfrom sklearn.metrics.pairwise import cosine_similarityfrom rich.box import HEAVYfrom rich.text import Textfrom rich.table import Tablefrom rich.console import Consolefrom huggingface_hub import hf_hub_download, snapshot_downloaddef running_in_colab() -> bool:    try:        import google.colab  # noqa: F401        return True    except ImportError:        return FalseIN_COLAB = running_in_colab()print(f"Environment: {'Colab' if IN_COLAB else 'local Jupyter'}")

### API keys and HF tokenSame pattern as the generation notebook: Colab pulls from Colab Secrets, local Jupyter pulls from the shell environment or a `.env` file. Query embedding at evaluation time calls the OpenAI API for the two OpenAI models, so `OPENAI_API_KEY` is required here too, not just in the generation notebook.

In [ ]:
if IN_COLAB:    from google.colab import userdata    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")else:    try:        from dotenv import load_dotenv        load_dotenv()    except ImportError:        pass    missing = [k for k in ("HF_TOKEN", "OPENAI_API_KEY") if not os.environ.get(k)]    if missing:        raise RuntimeError(            f"Missing required environment variable(s): {missing}. "            "Set them in your shell (export HF_TOKEN=...) or in a .env file "            "in the notebook's working directory before re-running this cell."        )print("HF_TOKEN set:", bool(os.environ.get("HF_TOKEN")))print("OPENAI_API_KEY set:", bool(os.environ.get("OPENAI_API_KEY")))

## ConfigurationSingle place to control which models, workloads, and content types this run covers, so re-running on a subset (e.g. just W3 while debugging) is a one-line edit.

In [ ]:
HF_REPO_ID = "Sakhiur/empirical-rag-paradigm-benchmark"MODELS_TO_BENCHMARK = [    "text-embedding-3-small",    "text-embedding-3-large",    "BAAI/bge-small-en-v1.5",    "Alibaba-NLP/gte-base-en-v1.5",]CATEGORIES = ["cs.AI", "cs.LG", "cs.IR", "cs.DB", "cs.SE"]WORKLOADS = ["w1", "w2", "w3"]CONTENT_TYPES = ["abstracts", "pdf_chunks"]METRICS = ["mrr", "recall@1", "recall@5", "ndcg@5", "ndcg@10"]LOCAL_DATA_ROOT = Path("data")OUTPUT_DIR = Path("benchmark_outputs")OUTPUT_DIR.mkdir(exist_ok=True)

## Download data from Hugging FacePulls the full `embeddings/` tree (all four models) and the `question_answer_pairs/` tree (all three workloads, both content types) in one snapshot download. Using `snapshot_download` with an `allow_patterns` filter rather than individual `hf_hub_download` calls per file, since the number of individual files (4 models x 2 content types x 5 categories, plus 3 workloads x 2 content types of queries) makes per-file calls slow and noisy.

In [ ]:
snapshot_download(    repo_id=HF_REPO_ID,    repo_type="dataset",    local_dir=LOCAL_DATA_ROOT,    allow_patterns=["embeddings/*", "question_answer_pairs/*"],)print("Download complete. Local tree:")for p in sorted(LOCAL_DATA_ROOT.rglob("*")):    if p.is_file():        print(" ", p.relative_to(LOCAL_DATA_ROOT))

## Load chunk embeddingsEach model's embeddings live at `embeddings/{model}/{content_type}/{category}.jsonl`, one JSON object per line: `{"chunk_id": ..., "embedding": [...]}`. Chunk IDs are already globally unique strings (bare arXiv ID for abstracts, `{arxiv_id}_p{page}` for pdf_chunks), so unlike the original reference notebooks this pipeline is based on, no chunk-ID remapping across files is needed. We load once per (model, content_type) across all five categories, since retrieval needs to rank across the whole corpus for that content type, not per-category.

In [ ]:
def load_chunk_embeddings(model_name: str, content_type: str) -> dict[str, np.ndarray]:    """Returns {chunk_id: embedding_vector} across all categories for one model + content type."""    embeddings: dict[str, np.ndarray] = {}    for category in CATEGORIES:        file_path = LOCAL_DATA_ROOT / "embeddings" / model_name / content_type / f"{category}.jsonl"        if not file_path.exists():            print(f"  WARNING: missing {file_path}, skipping this category for {model_name}")            continue        with open(file_path, "r") as f:            for line in f:                line = line.strip()                if not line:                    continue                record = json.loads(line)                embeddings[record["chunk_id"]] = np.array(record["embedding"], dtype=np.float32)    return embeddings

In [ ]:
# Pre-load every (model, content_type) combination once up front. This costs some# memory (four models x two content types x full corpus of vectors) but avoids# re-reading the same jsonl files repeatedly across the six evaluation slices below.chunk_embeddings_by_model_and_type: dict[tuple[str, str], dict[str, np.ndarray]] = {}for model_name in MODELS_TO_BENCHMARK:    for content_type in CONTENT_TYPES:        print(f"Loading {model_name} | {content_type} ...")        chunk_embeddings_by_model_and_type[(model_name, content_type)] = load_chunk_embeddings(            model_name, content_type        )        n = len(chunk_embeddings_by_model_and_type[(model_name, content_type)])        print(f"  {n} chunks loaded")

## Load queriesThe three workloads have different CSV schemas, which reflects genuinely different task structures rather than inconsistency:- **W1** (`chunk_id, category, bertopic_cluster, question, gold_chunk_id`): short-context factual lookup, one query maps to exactly one correct chunk.- **W2** (`pair_id, category, pair_type, chunk_id_a, chunk_id_b, question`): multi-hop reasoning, one query genuinely requires two source chunks, so it has two relevant chunks rather than one.- **W3** (`chunk_id, category, bertopic_cluster, query`): semantic similarity, one query maps to one anchor chunk, but unlike W1 the relevance is topical/conceptual rather than a single extractable factual answer.Because W2 has a structurally different ground-truth shape (two relevant chunks per query instead of one), we normalize all three into a common `{query_id: [relevant_chunk_ids]}` shape at load time. This lets one downstream evaluation function handle all three workloads without special-casing W2 deep inside the metric logic.

In [ ]:
def load_w1_queries(content_type: str) -> pd.DataFrame:    dfs = []    for category in CATEGORIES:        path = LOCAL_DATA_ROOT / "question_answer_pairs" / content_type / "w1" / f"{category}.csv"        if not path.exists():            print(f"  WARNING: missing {path}")            continue        df = pd.read_csv(path)        dfs.append(df)    combined = pd.concat(dfs, ignore_index=True)    combined["query_id"] = [f"w1_{i}" for i in range(len(combined))]    combined["query_text"] = combined["question"]    combined["relevant_chunk_ids"] = combined["gold_chunk_id"].apply(lambda x: [str(x)])    return combined[["query_id", "query_text", "relevant_chunk_ids", "category"]]def load_w2_queries(content_type: str) -> pd.DataFrame:    dfs = []    for category in CATEGORIES:        path = LOCAL_DATA_ROOT / "question_answer_pairs" / content_type / "w2" / f"{category}.csv"        if not path.exists():            print(f"  WARNING: missing {path}")            continue        df = pd.read_csv(path)        dfs.append(df)    combined = pd.concat(dfs, ignore_index=True)    combined["query_id"] = [f"w2_{i}" for i in range(len(combined))]    combined["query_text"] = combined["question"]    # Both chunk_id_a and chunk_id_b count as relevant: a retriever that surfaces    # either source chunk for a multi-hop question has found genuinely useful evidence,    # even if it hasn't found both halves of the hop.    combined["relevant_chunk_ids"] = combined.apply(        lambda row: [str(row["chunk_id_a"]), str(row["chunk_id_b"])], axis=1    )    return combined[["query_id", "query_text", "relevant_chunk_ids", "category"]]def load_w3_queries(content_type: str) -> pd.DataFrame:    dfs = []    for category in CATEGORIES:        path = LOCAL_DATA_ROOT / "question_answer_pairs" / content_type / "w3" / f"{category}.csv"        if not path.exists():            print(f"  WARNING: missing {path}")            continue        df = pd.read_csv(path)        dfs.append(df)    combined = pd.concat(dfs, ignore_index=True)    combined["query_id"] = [f"w3_{i}" for i in range(len(combined))]    combined["query_text"] = combined["query"]    combined["relevant_chunk_ids"] = combined["chunk_id"].apply(lambda x: [str(x)])    return combined[["query_id", "query_text", "relevant_chunk_ids", "category"]]QUERY_LOADERS = {"w1": load_w1_queries, "w2": load_w2_queries, "w3": load_w3_queries}

In [ ]:
# Queries themselves don't have per-model embeddings baked in the way the reference# notebooks assumed (there, question embeddings were nested inside the same JSON as# chunk embeddings). Here queries are plain text in CSV, so we embed them on the fly# for each model at evaluation time using the same embedding functions as generation.# For the two already-fully-generated OpenAI models this is a live API call; for the# open-source models it's a local forward pass, which is still fast for query-sized# batches (hundreds to low thousands of queries) without needing a GPU.from openai import OpenAIfrom sentence_transformers import SentenceTransformeropenai_client = OpenAI()_st_model_cache: dict[str, SentenceTransformer] = {}def embed_queries(model_name: str, texts: list[str]) -> np.ndarray:    if model_name.startswith("text-embedding-"):        response = openai_client.embeddings.create(model=model_name, input=texts)        return np.array([item.embedding for item in response.data], dtype=np.float32)    else:        if model_name not in _st_model_cache:            _st_model_cache[model_name] = SentenceTransformer(model_name)        vectors = _st_model_cache[model_name].encode(            texts, normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=False        )        return vectors.astype(np.float32)

## Build Qrels and RunsFollowing the same `ranx` pattern as the reference notebooks: `Qrels` is the ground-truth relevance mapping, `Run` is one model's ranked retrieval output per query. The one adaptation from the reference pattern is that `Qrels` here supports multiple relevant chunks per query (needed for W2), whereas the reference notebooks only ever had one.We cache query embeddings per (model, workload, content_type) since each is only computed once and reused for scoring against the full chunk set.

In [ ]:
def create_qrels(queries_df: pd.DataFrame) -> Qrels:    qrels_dict = {}    for _, row in queries_df.iterrows():        qrels_dict[row["query_id"]] = {cid: 1 for cid in row["relevant_chunk_ids"]}    return Qrels(qrels_dict)def create_run(    queries_df: pd.DataFrame,    query_embeddings: np.ndarray,    chunk_embeddings: dict[str, np.ndarray],    model_name: str,) -> Run:    chunk_ids = list(chunk_embeddings.keys())    chunk_matrix = np.stack([chunk_embeddings[cid] for cid in chunk_ids])    similarity_scores = cosine_similarity(query_embeddings, chunk_matrix)    run_dict = {}    for i, query_id in enumerate(queries_df["query_id"]):        run_dict[query_id] = {            cid: float(score) for cid, score in zip(chunk_ids, similarity_scores[i])        }    return Run(run_dict, name=model_name)

## Report formatting helperReused verbatim from the ranx reference notebook pattern (letter-coded models, bold-green best score, bold-red worst score, superscript letters marking statistically significant wins under Fisher's exact test at p < 0.05). Wrapped as a function here since it needs to run six times, once per evaluation slice, rather than once.

In [ ]:
def print_benchmark_report(report: Report) -> None:    model_to_letter = {}    for letter, name in zip(string.ascii_lowercase, report.model_names):        model_to_letter[name] = letter    report.rounding_digits = 4    cell_data = {}    max_widths = {metric: len(metric) for metric in report.metrics}    for current_model_name in report.model_names:        cell_data[current_model_name] = {}        for metric in report.metrics:            superscripts = ""            current_model_score = report.results[current_model_name][metric]            for other_model_name in report.model_names:                if current_model_name == other_model_name:                    continue                comparison_key = {current_model_name, other_model_name}                p_value = report.comparisons[comparison_key][metric]["p_value"]                other_model_score = report.results[other_model_name][metric]                if current_model_score > other_model_score and p_value < report.max_p:                    superscripts += model_to_letter[other_model_name]            score_str = f"{current_model_score:.{report.rounding_digits}f}"            sorted_superscripts = "".join(sorted(superscripts))            cell_data[current_model_name][metric] = (score_str, sorted_superscripts)            total_len = len(score_str)            if sorted_superscripts:                total_len += 1 + len(sorted_superscripts)            if total_len > max_widths[metric]:                max_widths[metric] = total_len    max_scores = {metric: -float("inf") for metric in report.metrics}    min_scores = {metric: float("inf") for metric in report.metrics}    for model_name in report.model_names:        for metric in report.metrics:            score = report.results[model_name][metric]            max_scores[metric] = max(max_scores[metric], score)            min_scores[metric] = min(min_scores[metric], score)    console = Console()    table = Table(show_header=True, header_style="bold bright_magenta", box=HEAVY)    table.add_column("#", style="dim")    table.add_column("Model", style="yellow")    for metric in report.metrics:        table.add_column(metric, justify="left", width=max_widths[metric])    table.add_column("Average", justify="right", style="bold cyan")    model_averages = {}    for model_name in report.model_names:        model_averages[model_name] = np.mean(            [report.results[model_name][metric] for metric in report.metrics]        )    sorted_models = sorted(model_averages.items(), key=lambda x: x[1], reverse=True)    best_model_name = sorted_models[0][0]    for model_name, avg_score in sorted_models:        model_name_cell = model_name        if model_name == best_model_name:            model_name_cell = f"[bold bright_green]{model_name}[/bold bright_green]"        row_data: list[Any] = [f"[dim]{model_to_letter[model_name]}[/dim]", model_name_cell]        for metric in report.metrics:            cell_text = Text()            score = report.results[model_name][metric]            score_part, superscript_part = cell_data[model_name][metric]            if score == max_scores[metric]:                cell_text.append(score_part, style="bold bright_green")            elif score == min_scores[metric]:                cell_text.append(score_part, style="bold bright_red")            else:                cell_text.append(score_part)            if superscript_part:                cell_text.append(" ")                cell_text.append(superscript_part, style="bold bright_cyan")            row_data.append(cell_text)        row_data.append(f"{avg_score:.4f}")        table.add_row(*row_data)    console.print(table)

## Run all six evaluation slicesFor each (workload, content_type) pair: load queries, embed them per model, build Qrels + Runs, and call `ranx.compare` with Fisher's exact test (same as the reference notebooks) to get statistically-annotated scores. Results are stored in a flat list of dicts as we go, so the whole thing can be dumped to a single tidy CSV afterward rather than six separate ones, which makes cross-slice comparison and plotting much easier later.`recall@1` and `recall@5` under multi-relevant qrels (W2) follow ranx's standard definition: the fraction of relevant items retrieved in the top-K, so a W2 query with one of its two relevant chunks in the top-5 scores 0.5 on recall@5, not 1.0. This is the correct behavior, since "found half the multi-hop evidence" is a genuinely weaker result than "found both halves," and Recall@K should reflect that.

In [ ]:
all_results_rows = []all_reports = {}  # (workload, content_type) -> Report, kept for the plotting section belowfor content_type in CONTENT_TYPES:    for workload in WORKLOADS:        print(f"\n{'=' * 70}")        print(f"  {workload.upper()} | {content_type}")        print(f"{'=' * 70}")        queries_df = QUERY_LOADERS[workload](content_type)        print(f"  {len(queries_df)} queries loaded")        qrels = create_qrels(queries_df)        runs = []        for model_name in MODELS_TO_BENCHMARK:            chunk_embeddings = chunk_embeddings_by_model_and_type[(model_name, content_type)]            if not chunk_embeddings:                print(f"  skipping {model_name}: no chunk embeddings loaded for {content_type}")                continue            query_embeddings = embed_queries(model_name, queries_df["query_text"].tolist())            run = create_run(queries_df, query_embeddings, chunk_embeddings, model_name)            runs.append(run)        if len(runs) < 2:            print(f"  skipping comparison: fewer than 2 models have embeddings for this slice")            continue        report = compare(qrels=qrels, runs=runs, metrics=METRICS, max_p=0.05, stat_test="fisher")        all_reports[(workload, content_type)] = report        print_benchmark_report(report)        for model_name in report.model_names:            row = {"workload": workload, "content_type": content_type, "model": model_name}            for metric in METRICS:                row[metric] = report.results[model_name][metric]            all_results_rows.append(row)

## Save results to CSVOne tidy CSV with a row per (workload, content_type, model), columns for each metric. This shape is directly usable for the thesis's results tables and for any further statistical testing (e.g. Wilcoxon signed-rank with Bonferroni correction across model pairs) without needing to re-run the benchmark.

In [ ]:
results_df = pd.DataFrame(all_results_rows)results_df = results_df.sort_values(["content_type", "workload", "model"]).reset_index(drop=True)csv_path = OUTPUT_DIR / "embedding_model_benchmark_results.csv"results_df.to_csv(csv_path, index=False)print(f"Saved {len(results_df)} rows to {csv_path}")results_df

## FiguresTwo views on the same results, chosen for different purposes:1. **Grouped bar chart per metric** — the standard way to show "which model wins" at a glance for a given metric, faceted by workload so W1/W2/W3 differences are visible rather than averaged away.2. **Heatmap of average score per (model, workload x content_type)** — a compact single-figure summary across all six slices at once, useful as a thesis figure that doesn't require six separate panels.Both read from `results_df` rather than re-running anything, so they're cheap to regenerate if you want to restyle them later.

In [ ]:
fig, axes = plt.subplots(1, len(METRICS), figsize=(5 * len(METRICS), 5), sharey=False)for content_type in CONTENT_TYPES:    subset = results_df[results_df["content_type"] == content_type]    for ax, metric in zip(axes, METRICS):        pivot = subset.pivot(index="workload", columns="model", values=metric)        pivot = pivot.reindex(WORKLOADS)        pivot.plot(kind="bar", ax=ax, legend=False)        ax.set_title(metric)        ax.set_xlabel("")        ax.tick_params(axis="x", rotation=0)    handles, labels = axes[0].get_legend_handles_labels()    fig.legend(handles, labels, loc="upper center", ncol=len(MODELS_TO_BENCHMARK), bbox_to_anchor=(0.5, 1.08))    fig.suptitle(f"Embedding model comparison — {content_type}", y=1.15, fontsize=14)    fig_path = OUTPUT_DIR / f"metric_comparison_{content_type}.png"    fig.savefig(fig_path, dpi=200, bbox_inches="tight")    print(f"Saved {fig_path}")    plt.show()    fig, axes = plt.subplots(1, len(METRICS), figsize=(5 * len(METRICS), 5), sharey=False)

In [ ]:
# Heatmap: rows = model, columns = workload x content_type slice, values = average# score across all metrics for that slice. Gives a single-glance summary figure.results_df["slice"] = results_df["content_type"] + " / " + results_df["workload"]results_df["avg_score"] = results_df[METRICS].mean(axis=1)heatmap_data = results_df.pivot(index="model", columns="slice", values="avg_score")fig, ax = plt.subplots(figsize=(10, 5))im = ax.imshow(heatmap_data.values, cmap="YlGnBu", aspect="auto")ax.set_xticks(range(len(heatmap_data.columns)))ax.set_xticklabels(heatmap_data.columns, rotation=45, ha="right")ax.set_yticks(range(len(heatmap_data.index)))ax.set_yticklabels(heatmap_data.index)for i in range(len(heatmap_data.index)):    for j in range(len(heatmap_data.columns)):        value = heatmap_data.values[i, j]        if not np.isnan(value):            ax.text(j, i, f"{value:.3f}", ha="center", va="center", fontsize=9)ax.set_title("Average score across all metrics, per model and evaluation slice")fig.colorbar(im, ax=ax, label="average score")fig.tight_layout()heatmap_path = OUTPUT_DIR / "model_slice_heatmap.png"fig.savefig(heatmap_path, dpi=200, bbox_inches="tight")print(f"Saved {heatmap_path}")plt.show()

## Overall model rankingAverages each model's score across all six slices to give a single overall ranking. This is the number to lead with when justifying the final model choice for the paradigm benchmark, with the per-slice breakdown above available as supporting detail if a reviewer asks whether the winner holds up across workload types, not just on average.

In [ ]:
overall_ranking = (    results_df.groupby("model")[METRICS]    .mean()    .assign(overall_avg=lambda df: df.mean(axis=1))    .sort_values("overall_avg", ascending=False))overall_ranking_path = OUTPUT_DIR / "overall_model_ranking.csv"overall_ranking.to_csv(overall_ranking_path)print(f"Saved {overall_ranking_path}")overall_ranking

## Next stepThe model with the strongest overall ranking above, checked against the per-slice breakdown to confirm it isn't winning only on one workload type, is the candidate to standardize on for the paradigm benchmark (pgvector vs MongoDB vs Qdrant). Once that choice is made, the retrieval benchmarking work moves from comparing embedding models (this notebook) to comparing database paradigms using the fixed, chosen model, which is the actual core contribution of the thesis.